# 📦 Notebook 1: Data Preparation
## Emotion-Conditioned Image Captioning

---

## ✅ What This Notebook Does
1. Mounts Google Drive (your permanent storage across sessions)
2. Downloads the **Flickr8k** dataset (images + captions)
3. Uses **GPT-2** to rewrite neutral captions into 5 emotional styles
4. Builds a vocabulary from the emotional captions
5. Saves everything to Google Drive for use in training

## ⚠️ Before You Run
- Make sure **Runtime → Change runtime type → T4 GPU** is selected
- You need a **Kaggle account** (free) to download Flickr8k — instructions below
- Run cells **one by one** and read the output before proceeding
- This notebook takes ~30–45 minutes total (mostly GPT-2 generation)

---

## Step 1 — Mount Google Drive

All outputs will be saved here so they survive Colab session resets.
Click the link that appears, sign in, copy the code, and paste it below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/EmotionCaptioning'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
print('✅ Drive mounted. Project folder created at:', PROJECT_DIR)

## Step 2 — Install Dependencies

In [ ]:
!pip install -q kaggle transformers nltk pycocoevalcap
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ Dependencies installed')

## Step 3 — Set Up Kaggle API

### How to get your Kaggle API key:
1. Go to https://www.kaggle.com → click your profile picture → **Account**
2. Scroll to **API** section → click **Create New Token**
3. A file `kaggle.json` will download — open it and copy the values below

Paste your username and key in the cell below:

In [ ]:
import os, json

# ✏️ PASTE YOUR KAGGLE CREDENTIALS HERE
KAGGLE_USERNAME = "your_kaggle_username"
KAGGLE_KEY      = "your_kaggle_api_key"

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials saved')

## Step 4 — Download Flickr8k

This downloads ~1GB of images and caption files. Takes 3–5 minutes.

> **Note:** You must accept the dataset terms on Kaggle first.
> Visit: https://www.kaggle.com/datasets/adityajn105/flickr8k and click **Download**.

In [ ]:
import os

DATA_RAW = '/content/flickr8k_raw'
os.makedirs(DATA_RAW, exist_ok=True)

# Download and unzip
!kaggle datasets download -d adityajn105/flickr8k -p {DATA_RAW} --unzip -q
print('✅ Download complete')
!ls {DATA_RAW}

## Step 5 — Parse and Inspect Captions

In [ ]:
import os, glob, json, random
import pandas as pd

# Flickr8k caption file is usually captions.txt with format: image#N\tcaption
caption_file = os.path.join(DATA_RAW, 'captions.txt')
if not os.path.exists(caption_file):
    # Try alternate paths
    candidates = glob.glob(DATA_RAW + '/**/*.txt', recursive=True)
    print('Found text files:', candidates)
    caption_file = candidates[0]  # Adjust if needed

df = pd.read_csv(caption_file)
print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
print(df.head(10))

In [ ]:
# Standardize column names (handle different Flickr8k formats)
if 'image' in df.columns and 'caption' in df.columns:
    df_caps = df.rename(columns={'image': 'image_id', 'caption': 'caption'})
elif 'image_id' in df.columns:
    df_caps = df
else:
    # Tab-separated format: image#0\tcaption
    df_caps = pd.read_csv(caption_file, sep='\t', names=['image_id', 'caption'])
    df_caps['image_id'] = df_caps['image_id'].apply(lambda x: x.split('#')[0])

# Keep one caption per image for simplicity (first caption)
df_unique = df_caps.drop_duplicates(subset='image_id', keep='first').copy()
df_unique = df_unique[df_unique['caption'].notna()].reset_index(drop=True)

# Verify images exist
img_dir = os.path.join(DATA_RAW, 'Images')
if not os.path.isdir(img_dir):
    img_dir = DATA_RAW  # images may be in root

df_unique['img_path'] = df_unique['image_id'].apply(lambda x: os.path.join(img_dir, x))
df_unique = df_unique[df_unique['img_path'].apply(os.path.exists)].reset_index(drop=True)

print(f'✅ {len(df_unique)} unique images with captions found')
print(df_unique[['image_id','caption']].head(5))

## Step 6 — Generate Emotion Captions with GPT-2

This is the **core data augmentation step**. We take neutral Flickr8k captions and rewrite them into 5 emotional styles using GPT-2.

The 5 emotions are: `joyful`, `sad`, `tense`, `romantic`, `humorous`

**This takes ~20–35 minutes on free Colab T4.** Get a coffee ☕

> We use **sampling-based generation** (temperature=0.9) for diverse outputs, not greedy decoding.

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

tokenizer_gpt2 = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token
model_gpt2 = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
model_gpt2.eval()
print('✅ GPT-2 loaded')

In [ ]:
EMOTIONS = ['joyful', 'sad', 'tense', 'romantic', 'humorous']
EMOTION_TO_ID = {e: i for i, e in enumerate(EMOTIONS)}

EMOTION_PROMPTS = {
    'joyful':    'Rewrite this image caption in a joyful, upbeat, and cheerful tone: "{caption}" Rewritten: ',
    'sad':       'Rewrite this image caption in a sad, melancholic, and somber tone: "{caption}" Rewritten: ',
    'tense':     'Rewrite this image caption in a tense, dramatic, and suspenseful tone: "{caption}" Rewritten: ',
    'romantic':  'Rewrite this image caption in a romantic, warm, and tender tone: "{caption}" Rewritten: ',
    'humorous':  'Rewrite this image caption in a humorous, funny, and witty tone: "{caption}" Rewritten: ',
}

def generate_emotion_caption(neutral_caption, emotion, max_new_tokens=40):
    prompt = EMOTION_PROMPTS[emotion].format(caption=neutral_caption.strip())
    inputs = tokenizer_gpt2(prompt, return_tensors='pt', truncation=True, max_length=200).to(device)
    prompt_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        output = model_gpt2.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_p=0.92,
            repetition_penalty=1.3,
            pad_token_id=tokenizer_gpt2.eos_token_id,
        )
    generated = output[0][prompt_len:]
    text = tokenizer_gpt2.decode(generated, skip_special_tokens=True)
    # Keep only the first sentence
    for sep in ['.', '!', '?', '\n']:
        if sep in text:
            text = text[:text.index(sep)+1]
            break
    return text.strip()

# Quick sanity check
test_cap = df_unique['caption'].iloc[0]
print('Original:', test_cap)
for e in EMOTIONS:
    print(f'{e:12s}:', generate_emotion_caption(test_cap, e))

In [ ]:
from tqdm.auto import tqdm

# Use all 8k images (or reduce N_SAMPLES if Colab disconnects)
N_SAMPLES = len(df_unique)  # Set to e.g. 2000 if you want faster runs
df_sample = df_unique.iloc[:N_SAMPLES].copy()

records = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc='Generating emotion captions'):
    for emotion in EMOTIONS:
        emo_cap = generate_emotion_caption(row['caption'], emotion)
        records.append({
            'image_id': row['image_id'],
            'img_path': row['img_path'],
            'neutral_caption': row['caption'],
            'emotion': emotion,
            'emotion_id': EMOTION_TO_ID[emotion],
            'emotion_caption': emo_cap,
        })

df_emotion = pd.DataFrame(records)
print(f'✅ Generated {len(df_emotion)} (image, emotion, caption) triples')
print(df_emotion.head(10))

## Step 7 — Build Vocabulary

In [ ]:
from collections import Counter
import re

def tokenize(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s']", '', text)
    return text.split()

# Count word frequencies
counter = Counter()
for cap in df_emotion['emotion_caption']:
    counter.update(tokenize(cap))

MIN_FREQ = 2  # Words appearing less than this are replaced with <UNK>

vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
for word, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

idx_to_word = {v: k for k, v in vocab.items()}

print(f'✅ Vocabulary size: {len(vocab)} words')
print('Sample vocab:', list(vocab.items())[:20])

## Step 8 — Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split by image_id to avoid data leakage
all_image_ids = df_emotion['image_id'].unique()
train_ids, temp_ids = train_test_split(all_image_ids, test_size=0.2, random_state=42)
val_ids, test_ids   = train_test_split(temp_ids, test_size=0.5, random_state=42)

df_train = df_emotion[df_emotion['image_id'].isin(train_ids)].reset_index(drop=True)
df_val   = df_emotion[df_emotion['image_id'].isin(val_ids)].reset_index(drop=True)
df_test  = df_emotion[df_emotion['image_id'].isin(test_ids)].reset_index(drop=True)

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

## Step 9 — Save Everything to Google Drive

In [ ]:
import pickle

df_train.to_csv(f'{PROJECT_DIR}/data/train.csv', index=False)
df_val.to_csv(f'{PROJECT_DIR}/data/val.csv', index=False)
df_test.to_csv(f'{PROJECT_DIR}/data/test.csv', index=False)

with open(f'{PROJECT_DIR}/data/vocab.pkl', 'wb') as f:
    pickle.dump({'vocab': vocab, 'idx_to_word': idx_to_word, 'emotions': EMOTIONS}, f)

# Save config for other notebooks
config = {
    'PROJECT_DIR': PROJECT_DIR,
    'IMG_DIR': img_dir,
    'VOCAB_SIZE': len(vocab),
    'EMOTIONS': EMOTIONS,
    'EMOTION_TO_ID': EMOTION_TO_ID,
    'N_EMOTIONS': len(EMOTIONS),
    'EMBED_DIM': 64,
    'HIDDEN_DIM': 512,
    'VISUAL_DIM': 2048,   # ResNet-50
    'FUSED_DIM': 512,
    'MAX_SEQ_LEN': 40,
}
with open(f'{PROJECT_DIR}/data/config.pkl', 'wb') as f:
    pickle.dump(config, f)

print('✅ All data saved to Google Drive!')
print(f'  Train: {len(df_train)} samples')
print(f'  Val:   {len(df_val)} samples')
print(f'  Test:  {len(df_test)} samples')
print(f'  Vocab: {len(vocab)} words')

## Step 10 — Quick Data Sanity Check

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Show a few examples
sample = df_train.sample(5, random_state=0)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (_, row) in zip(axes, sample.iterrows()):
    img = Image.open(row['img_path']).convert('RGB')
    ax.imshow(img)
    ax.set_title(f"{row['emotion']}\n{row['emotion_caption'][:60]}...", fontsize=7, wrap=True)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/data/sample_viz.png', dpi=100)
plt.show()
print('✅ Sample visualization saved')

In [ ]:
# Emotion distribution check
df_train['emotion'].value_counts().plot(kind='bar', title='Emotion distribution (train)')
plt.tight_layout()
plt.show()
print('✅ Distribution looks balanced (each emotion ~20%)')

---
## ✅ Notebook 1 Complete!

### What was saved to Google Drive:
| File | Contents |
|------|----------|
| `data/train.csv` | Training triples (image_id, emotion, caption) |
| `data/val.csv`   | Validation triples |
| `data/test.csv`  | Test triples |
| `data/vocab.pkl` | Word → index mapping |
| `data/config.pkl`| Model hyperparameter config |

### ➡️ Next Step
Open **Notebook 2: Model Training** and run it.
The images stay in `/content/flickr8k_raw` — if your session resets, re-run the download cell.